In [ ]:
# Portable project paths. Set TLS_PROJECT_ROOT to the directory containing the input data.
import os
from pathlib import Path
PROJECT_ROOT = Path(os.environ.get("TLS_PROJECT_ROOT", ".")).resolve()


In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd

adata = sc.read_h5ad("/data/beifen/zhongmin/slide-tag/H5AD格式数据/slide-tag肺_with_scanvi_加上补测数据_对3级淋巴结构进行分类_186_2025_12_30.h5ad")
print(adata)

In [ ]:
num_unique_samples = adata.obs['tls_degrow_id'].nunique()
print(f"唯一样本数量: {num_unique_samples}")

In [ ]:
# ============================================
# Robust tumor zoning + robust TLS region assignment + plotting
# （完整可运行版本：修复“无肿瘤样本 dist_out=0 导致 TLS 误判 normal_adjacent”的逻辑漏洞）
# ============================================

# ---------- Imports ----------
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from scipy.ndimage import (
    distance_transform_edt,
    binary_dilation, binary_erosion,
    binary_closing, binary_opening,
    label as cc_label
)

# ---------- 小工具 ----------
def _get_spatial(adata, key_candidates=("spatial", "X_spatial")):
    """
    返回 (使用的键名, 坐标数组)
    优先使用 adata.obsm['spatial']，否则回退到 'X_spatial'
    """
    for k in key_candidates:
        if k in adata.obsm:
            XY = np.asarray(adata.obsm[k], dtype=float)
            if XY.ndim != 2 or XY.shape[1] < 2:
                raise ValueError(f"obsm['{k}'] 应为 (n,2) 或以上，当前形状={XY.shape}")
            return k, XY[:, :2]
    raise KeyError(f"找不到空间坐标列，已尝试 {key_candidates}")

def _circular_kernel(radius_px: int):
    r = int(max(1, round(radius_px)))
    y, x = np.ogrid[-r:r+1, -r:r+1]
    return (x*x + y*y) <= (r*r)

def _tls_ids_mask(series):
    s = series.astype("string").str.strip()
    mask = (~s.isna()) & s.ne("") & s.str.lower().ne("none")
    return mask, s

def _xy_to_grid(xy_um, xmin, ymin, step, W, H):
    xs = np.clip(np.round((xy_um[:, 0] - xmin)/step).astype(int), 0, W-1)
    ys = np.clip(np.round((xy_um[:, 1] - ymin)/step).astype(int), 0, H-1)
    return xs, ys

# ---------- 1) 构建肿瘤四区（稳健） ----------
def build_tumor_zones_robust(
    adata,
    *,
    celltype_col="celltype_4_ZZM",
    malignant_keywords=("Malignant cells",),
    sample_col="sample",
    microns_per_unit=1.0,   # 坐标单位→微米的换算；若坐标已是微米，设为 1.0
    step_um=3.0,            # 栅格分辨率（越小越精细）
    open_um=0.0,            # opening 半径（μm）；设 0 关闭
    dilate_um=15.0,         # dilation 半径（μm）
    closing_um=25.0,        # closing 半径（μm）
    min_cc_um2=200.0,       # 最小连通域面积阈值（μm²）
    core_width_um=40.0,     # 肿瘤内部 ≥ 该最近边界距离 → core
    adj_width_um=40.0,      # 肿瘤外部 ≤ 该最近边界距离 → normal_adjacent
    verbose=True,
):
    """
    返回：
      zones_per_cell: 长度 n_obs 的 object 数组
        ('tumor_core' / 'tumor_margin' / 'normal_adjacent' / 'normal_distant')
      geo_per_sample: dict[sample] = { mask, dist_in, dist_out, xmin,ymin,step,W,H }
        供后续 TLS 判定使用

    ★关键修复：
      当某样本 tumor mask 为空（无恶性/被形态学过滤空）时：
      dist_out 必须设为 +inf，而不是 0，否则 TLS 会被误判为 normal_adjacent
    """
    spatial_key, XY_all_raw = _get_spatial(adata)
    XY_all = XY_all_raw * float(microns_per_unit)

    # 识别恶性
    mk = tuple(k.lower() for k in malignant_keywords)
    malignant_mask_all = adata.obs[celltype_col].astype(str).str.lower().map(
        lambda s: any(k in s for k in mk)
    ).to_numpy()

    samples = adata.obs[sample_col].astype(str).to_numpy()
    uniq_s = pd.Index(samples).unique()

    out = np.full(adata.n_obs, "normal_distant", dtype=object)
    geo = {}

    step = float(step_um)
    rad_open  = max(0, int(round(open_um/step)))
    rad_dil   = max(0, int(round(dilate_um/step)))
    rad_close = max(0, int(round(closing_um/step)))
    min_cc_px = int(round(max(min_cc_um2, 0) / (step*step)))

    for s in uniq_s:
        idx = np.where(samples == s)[0]
        XY  = XY_all[idx]
        mal = malignant_mask_all[idx]
        if idx.size == 0:
            continue

        # 适度外扩边界
        pad = 200.0
        xmin, ymin = XY.min(axis=0) - pad
        xmax, ymax = XY.max(axis=0) + pad
        W = int(np.ceil((xmax - xmin)/step)) + 1
        H = int(np.ceil((ymax - ymin)/step)) + 1

        tumor = np.zeros((H, W), dtype=bool)

        # --- 用恶性点构建初始 mask ---
        if mal.any():
            xs, ys = _xy_to_grid(XY[mal], xmin, ymin, step, W, H)
            tumor[ys, xs] = True
            area0 = int(tumor.sum())

            # opening（可选）
            if rad_open > 0:
                tumor = binary_opening(tumor, structure=_circular_kernel(rad_open))

            # dilation + closing（平滑&连接）
            if rad_dil > 0:
                tumor = binary_dilation(tumor, structure=_circular_kernel(rad_dil))
            if rad_close > 0:
                tumor = binary_closing(tumor, structure=_circular_kernel(rad_close))

            # 过滤小连通域
            if min_cc_px > 0 and tumor.any():
                lab, nlab = cc_label(tumor)
                sizes_px = np.bincount(lab.ravel(), minlength=nlab+1)
                keep_mask = np.zeros_like(tumor, dtype=bool)
                for k in range(1, nlab+1):
                    if sizes_px[k] >= min_cc_px:
                        keep_mask |= (lab == k)
                tumor = keep_mask

            # 自救：若被撸空，回退到“仅轻度膨胀”
            if (not tumor.any()) and area0 > 0:
                tumor = np.zeros((H, W), dtype=bool)
                tumor[ys, xs] = True
                tumor = binary_dilation(
                    tumor,
                    structure=_circular_kernel(max(1, int(round(6/step))))
                )

        # --- 计算距离场 & 四区 ---
        if tumor.any():
            dist_in  = distance_transform_edt(tumor)   * step
            dist_out = distance_transform_edt(~tumor) * step

            core_grid   = dist_in >= float(core_width_um)
            margin_grid = tumor & (~core_grid)
            adj_grid    = (~tumor) & (dist_out <= float(adj_width_um))
        else:
            # ★关键修复：没有肿瘤时 dist_out 应为 +inf，避免 TLS 被误判 adjacent
            dist_in  = np.zeros((H, W), dtype=float)
            dist_out = np.full((H, W), np.inf, dtype=float)
            core_grid = np.zeros((H, W), dtype=bool)
            margin_grid = np.zeros((H, W), dtype=bool)
            adj_grid = np.zeros((H, W), dtype=bool)

        # --- 映射回细胞 ---
        xs_all, ys_all = _xy_to_grid(XY, xmin, ymin, step, W, H)
        lab_s = np.full(idx.size, "normal_distant", dtype=object)
        m = adj_grid[ys_all, xs_all];     lab_s[m] = "normal_adjacent"
        m = margin_grid[ys_all, xs_all];  lab_s[m] = "tumor_margin"
        m = core_grid[ys_all, xs_all];    lab_s[m] = "tumor_core"
        out[idx] = lab_s

        geo[s] = dict(mask=tumor, dist_in=dist_in, dist_out=dist_out,
                      xmin=xmin, ymin=ymin, step=step, W=W, H=H)

        if verbose:
            vc = pd.Series(lab_s).value_counts()
            print(f"[{s}] cells={idx.size} | malignant={int(mal.sum())} | "
                  f"core={vc.get('tumor_core',0)} margin={vc.get('tumor_margin',0)} "
                  f"adj={vc.get('normal_adjacent',0)} distant={vc.get('normal_distant',0)}")

    return out, geo

# ---------- 2) TLS 的稳健分区（质心 + 重叠率纠偏） ----------
def assign_tls_region_robust(
    adata,
    geo_per_sample,
    *,
    sample_col="sample",
    tls_id_col="tls_degrow_id_sample",
    tls_region_col="tls_region_tlsmode",
    core_width_um=40.0,
    adj_width_um=40.0,
    flip_threshold=0.25,
    verbose=True,
):
    """
    输出列 tls_region_col：同一 TLS 的所有成员细胞用同一分区（TLS-level）
    ★关键修复：
      若该 sample tumor mask 为空（np.any(mask)==False），该样本所有 TLS 强制 normal_distant
    """
    # TLS id 有效性掩膜（numpy bool）
    ok_mask, tls_series = _tls_ids_mask(adata.obs[tls_id_col])
    ok_mask_np = ok_mask.to_numpy(dtype=bool)

    # sample（numpy）
    samples = adata.obs[sample_col].astype(str).to_numpy()
    uniq_s = pd.Index(samples).unique()

    # 空间坐标
    _, XY_all = _get_spatial(adata)

    tls2region = {}

    for s in uniq_s:
        m_s = (samples == s)
        m_tls = m_s & ok_mask_np
        if not m_tls.any():
            continue

        geom = geo_per_sample.get(s, None)
        if geom is None:
            # 没有几何信息：TLS 全部 normal_distant
            for tls_id in pd.Index(tls_series[m_tls]).dropna().unique():
                tls2region[str(tls_id).strip()] = "normal_distant"
            continue

        mask = geom["mask"]
        dist_in = geom["dist_in"]
        dist_out = geom["dist_out"]
        xmin, ymin, step, W, H = geom["xmin"], geom["ymin"], geom["step"], geom["W"], geom["H"]

        # ★关键修复：tumor mask 为空 → 该样本所有 TLS 强制 normal_distant
        if (mask is None) or (not np.any(mask)):
            for tls_id in pd.Index(tls_series[m_tls]).dropna().unique():
                tls2region[str(tls_id).strip()] = "normal_distant"
            if verbose:
                print(f"[{s}] tumor mask empty -> all TLS set to normal_distant")
            continue

        # 遍历该样本 TLS
        for tls_id in pd.Index(tls_series[m_tls]).dropna().unique():
            tls_id_str = str(tls_id).strip()
            if tls_id_str == "" or tls_id_str.lower() in {"none", "nan", "<na>"}:
                continue

            # 关键：确保 m_this 为 numpy bool
            m_this = tls_series.eq(tls_id).fillna(False).to_numpy(dtype=bool) & m_s
            XY = XY_all[m_this]
            if XY.shape[0] == 0:
                continue

            # 1) 质心粗分类
            centroid = XY.mean(axis=0)
            xg, yg = _xy_to_grid(centroid.reshape(1, 2), xmin, ymin, step, W, H)
            xg, yg = int(xg[0]), int(yg[0])
            xg = np.clip(xg, 0, W-1)
            yg = np.clip(yg, 0, H-1)

            inside_tumor = bool(mask[yg, xg])
            if inside_tumor:
                region0 = "tumor_core" if dist_in[yg, xg] >= float(core_width_um) else "tumor_margin"
            else:
                region0 = "normal_adjacent" if dist_out[yg, xg] <= float(adj_width_um) else "normal_distant"

            # 2) 与肿瘤掩膜的重叠率纠偏（仅对被判到肿瘤内的情况）
            xs, ys = _xy_to_grid(XY, xmin, ymin, step, W, H)
            overlap_frac = float(mask[ys, xs].sum()) / float(len(xs))

            region = region0
            if region0 in ("tumor_core", "tumor_margin") and overlap_frac < float(flip_threshold):
                region = "normal_adjacent" if dist_out[yg, xg] <= float(adj_width_um) else "normal_distant"
                if verbose:
                    print(f"  [flip] {tls_id_str} @ {s}: {region0} → {region} (overlap={overlap_frac:.2f})")

            tls2region[tls_id_str] = region

    # 映射回细胞（同一 TLS 统一标签）
    tls_id_series_str = adata.obs[tls_id_col].astype("string")
    region_col = pd.Series(
        [tls2region.get(str(t).strip(), np.nan) for t in tls_id_series_str],
        index=adata.obs_names,
        dtype="string",
    )
    adata.obs[tls_region_col] = region_col

    if verbose:
        ser = region_col.dropna()
        print("\n[Summary] TLS-level regions (mapped to cells):")
        print(ser.value_counts().to_string())

    tls_map_df = pd.DataFrame({"tls_degrow_id": list(tls2region.keys()),
                               "tls_region": list(tls2region.values())})
    return tls_map_df

# ---------- 3) TLS 级表 & 绘图 ----------
def _mode_or_nan(series: pd.Series):
    """
    返回分组众数；全缺或全是空白/None/na/nan 时返回 np.nan。
    """
    s = series.astype("string").str.strip()
    s = s.mask(s.isna() | s.eq("") | s.str.lower().isin({"none", "na", "nan"})).dropna()
    if s.empty:
        return np.nan
    m = s.mode()
    return m.iloc[0] if not m.empty else np.nan

def tls_level_table(
    adata,
    tls_id_col="tls_degrow_id_sample",
    region_col="tls_region_tlsmode",
    fenlei_col="tls_degrow_id_fenlei",
    drop_region_none=True,
):
    """
    把细胞级 obs 聚合到“TLS 级别”（每个 tls_id 一行）
    返回列：tls_id, tls_region(众数), fenlei(众数)
    —— 写 h5ad 兼容：统一用 pandas string 再 fillna
    """
    need = (tls_id_col, region_col, fenlei_col)
    if not all(c in adata.obs.columns for c in need):
        missing = [c for c in need if c not in adata.obs.columns]
        raise KeyError(f"adata.obs 缺少列：{missing}")

    df = adata.obs[[tls_id_col, region_col, fenlei_col]].copy()

    # 清洗 tls_id
    tls_id = df[tls_id_col].astype("string").str.strip()
    keep = (~tls_id.isna()) & tls_id.ne("") & tls_id.str.lower().ne("none")
    df = df.loc[keep].copy()
    df[tls_id_col] = tls_id.loc[keep]

    df[region_col] = df[region_col].astype("string")
    df[fenlei_col] = df[fenlei_col].astype("string")

    g = df.groupby(by=df[tls_id_col], observed=True, sort=False)
    out = g.agg({region_col: _mode_or_nan, fenlei_col: _mode_or_nan}).reset_index()

    out[region_col] = out[region_col].astype("string")
    out[fenlei_col] = out[fenlei_col].astype("string").fillna("Unknown")

    if drop_region_none:
        bad = out[region_col].isna() | out[region_col].eq("") | out[region_col].str.lower().eq("none")
        out = out.loc[~bad].copy()

    return out

def plot_tls_region_by_fenlei_TLS(
    adata,
    tls_id_col="tls_degrow_id_sample",
    region_col="tls_region_tlsmode",
    fenlei_col="tls_degrow_id_fenlei",
    categories=None,
    normalize=True,
    sort_regions=None,
    figsize=(10, 4),
    show_values=True,
):
    """
    简版：用于快速检查；如需论文级美化请用你后续的 publication 版本绘图函数。
    """
    tls_level = tls_level_table(
        adata,
        tls_id_col=tls_id_col,
        region_col=region_col,
        fenlei_col=fenlei_col,
        drop_region_none=True,
    )

    counts = pd.crosstab(tls_level[region_col], tls_level[fenlei_col])

    if categories is not None:
        keep = [c for c in categories if c in counts.columns]
        counts = counts[keep] if keep else counts.iloc[:, 0:0]

    if sort_regions == "count_desc" and counts.shape[0] > 0:
        counts = counts.loc[counts.sum(axis=1).sort_values(ascending=False).index]
    elif sort_regions == "alpha":
        counts = counts.sort_index()

    if normalize and counts.shape[0] > 0:
        denom = counts.sum(axis=1).replace(0, np.nan)
        plot_df = counts.div(denom, axis=0).fillna(0.0)
        y_label = "占比"; to_percent = True
    else:
        plot_df = counts
        y_label = "TLS 数"; to_percent = False

    palette = {
        "Mature":      "#59A14F",
        "Activating":  "#F28E2B",
        "Conforming":  "#4E79A7",
        "Deviating":   "#E15759",
        "undetermined":"#BAB0AC",
        "Unknown":     "#9C755F",
    }
    colors = [palette.get(c, "#888888") for c in plot_df.columns]

    fig, ax = plt.subplots(figsize=figsize)
    x = np.arange(plot_df.shape[0])
    bottom = np.zeros(plot_df.shape[0])

    for i, c in enumerate(plot_df.columns):
        vals = plot_df[c].values
        ax.bar(x, vals, bottom=bottom, label=c, color=colors[i], edgecolor="white", linewidth=0.6)
        if show_values and normalize:
            for xi, b, v in zip(x, bottom, vals):
                if v >= 0.05:
                    ax.text(xi, b + v/2, f"{v*100:.0f}%", ha="center", va="center",
                            fontsize=9, color="white")
        bottom += vals

    ax.set_xticks(x)
    ax.set_xticklabels(plot_df.index, rotation=25, ha="right")
    if to_percent:
        ax.set_ylim(0, 1)
        ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y*100:.0f}%"))
    ax.set_ylabel(y_label)
    ax.set_xlabel(region_col)
    ax.set_title(f"{region_col} × {fenlei_col}（TLS级）{'占比' if normalize else '计数'}")
    if plot_df.shape[1] > 0:
        ax.legend(title=fenlei_col, bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)

    plt.tight_layout()
    plt.show()
    return fig, plot_df, counts

# ---------- 4) 一键执行（示例调用） ----------
# 1) 先根据恶性细胞构建四区（细胞级）
zones, geo = build_tumor_zones_robust(
    adata,
    celltype_col="celltype_4_ZZM",
    malignant_keywords=("Malignant cells",),
    sample_col="sample",
    microns_per_unit=1.0,
    step_um=3.0,
    open_um=0.0, dilate_um=15.0, closing_um=25.0,
    min_cc_um2=200.0,
    core_width_um=40.0,
    adj_width_um=40.0,
    verbose=True
)
adata.obs["lymph_region_type"] = zones  # 细胞级四区（可视化/参考）

# 2) TLS 统一分区（TLS-level），并映射回细胞
tls_map = assign_tls_region_robust(
    adata, geo,
    sample_col="sample",
    tls_id_col="tls_degrow_id_sample",
    tls_region_col="tls_region_tlsmode",
    core_width_um=40.0,
    adj_width_um=40.0,
    flip_threshold=0.25,
    verbose=True
)

# 3) 快速检查：TLS 类型在四区的占比/数目
fig, plot_df, counts = plot_tls_region_by_fenlei_TLS(
    adata,
    tls_id_col="tls_degrow_id_sample",
    region_col="tls_region_tlsmode",
    fenlei_col="tls_degrow_id_fenlei",
    categories=["Mature", "Conforming", "Deviating", "Activating"],
    normalize=True,
    sort_regions="count_desc",
)


In [ ]:
# =========================
# Pretty spatial plotting (fixed point size)
# =========================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

def plot_lymph_regions_magazine(
    adata,
    *,
    sample_col="sample",
    spatial_key="spatial",
    region_col="lymph_region_type",
    # 颜色（按你的指定）
    palette={
        "tumor_core":      "#E95C59",
        "tumor_margin":    "#E59CC4",
        "normal_adjacent": "#F1BB72",
        "normal_distant":  "#53A85F",
    },
    # 分面与标注
    n_cols=3,
    figsize_per_panel=4.8,
    title_fontsize=12,
    annotate_counts=True,
    # 点样式（固定大小）
    point_size=10.0,            # ← 固定点大小（和 sc.pl.embedding(size=10) 类似）
    adaptive_point_size=False,  # ← 设 True 才会按样本点数自适应
    alpha=0.95,
    # 速度&质量
    rasterize_threshold=150_000, # 单面板点数超过这个阈值时开启栅格化
    # 导出
    suptitle="Spatial regions per sample",
    savepath=None,
    dpi=300,
    invert_y=False              # 若想“像切片图片”显示，可置 True 倒置 y 轴
):
    # 检查必需字段
    if sample_col not in adata.obs.columns:
        raise KeyError(f"adata.obs 缺少列：{sample_col}")
    if region_col not in adata.obs.columns:
        raise KeyError(f"adata.obs 缺少列：{region_col}")
    if spatial_key not in adata.obsm:
        raise KeyError(f"adata.obsm 缺少键：{spatial_key}")
    XY = np.asarray(adata.obsm[spatial_key], dtype=float)
    if XY.shape[1] != 2:
        raise ValueError("空间坐标必须是二维 (n_cells x 2)")

    # 样本顺序固定，便于复现
    samples = pd.Index(adata.obs[sample_col].astype(str)).unique().sort_values()

    # 简洁风格
    plt.rcParams.update({
        "font.size": 10,
        "axes.facecolor": "white",
        "axes.edgecolor": "none",
        "axes.titlesize": title_fontsize,
        "legend.frameon": False,
    })

    # 计算子图布局
    import math
    n = len(samples)
    n_cols = max(1, int(n_cols))
    n_rows = math.ceil(n / n_cols)
    fig, axs = plt.subplots(
        n_rows, n_cols,
        figsize=(figsize_per_panel*n_cols, figsize_per_panel*n_rows),
        constrained_layout=False
    )
    if n_rows == 1 and n_cols == 1:
        axs = np.array([[axs]])
    elif n_rows == 1:
        axs = np.array([axs])
    elif n_cols == 1:
        axs = np.array([[a] for a in axs])

    # 绘制顺序（先低优先级，后高优先级）
    draw_order = ["normal_distant", "normal_adjacent", "tumor_margin", "tumor_core"]

    for i, s in enumerate(samples):
        r, c = divmod(i, n_cols)
        ax = axs[r, c]
        mask_s = (adata.obs[sample_col].astype(str).values == s)
        XY_s = XY[mask_s]
        reg_s = adata.obs.loc[mask_s, region_col].astype(str).values

        # 点大小：固定 or 自适应
        n_pts = XY_s.shape[0]
        if adaptive_point_size:
            if   n_pts <= 10_000: ps = 3.0
            elif n_pts <= 30_000: ps = 2.2
            elif n_pts <= 80_000: ps = 1.6
            else:                 ps = 1.1
        else:
            ps = float(point_size)  # ← 固定值

        rasterized = (n_pts >= rasterize_threshold)

        for key in draw_order:
            m = (reg_s == key)
            if not m.any():
                continue
            ax.scatter(
                XY_s[m, 0], XY_s[m, 1],
                s=ps,
                c=palette.get(key, "#999999"),
                alpha=alpha,
                linewidths=0,
                rasterized=rasterized,
            )

        ax.set_title(f"{s}", pad=4, fontweight="bold")
        ax.set_aspect("equal", adjustable="datalim")
        ax.set_xticks([]); ax.set_yticks([])
        if invert_y:
            ax.invert_yaxis()
        for spine in ax.spines.values():
            spine.set_visible(False)

        if annotate_counts:
            vc = pd.Series(reg_s).value_counts()
            lines = [f"n={n_pts}"]
            for k in draw_order[::-1]:
                if k in vc.index:
                    lines.append(f"{k}: {vc[k]}")
            ax.text(
                0.02, 0.98, "\n".join(lines),
                transform=ax.transAxes,
                ha="left", va="top",
                fontsize=8, color="#222222",
                bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="none", alpha=0.85)
            )

    # 隐藏空子图
    total_axes = n_rows * n_cols
    for j in range(n, total_axes):
        r, c = divmod(j, n_cols)
        axs[r, c].axis("off")

    # 全局图例放右侧
    legend_items, legend_labels = [], []
    for k in ["tumor_core","tumor_margin","normal_adjacent","normal_distant"]:
        legend_items.append(Line2D([0], [0], marker='o', linestyle='',
                                   markersize=6, color=palette.get(k, "#999999")))
        legend_labels.append(k)
    fig.legend(
        legend_items, legend_labels,
        title="Region",
        loc="center left",
        bbox_to_anchor=(1.01, 0.5),
        frameon=False
    )

    if suptitle:
        fig.suptitle(suptitle, y=0.995, fontsize=14, fontweight="bold")

    plt.tight_layout(rect=(0, 0, 0.96, 0.98))
    if savepath is not None:
        fig.savefig(savepath, dpi=dpi, bbox_inches="tight")
    return fig

# ====== 使用示例（固定大小 = 10）======
fig = plot_lymph_regions_magazine(
    adata,
    sample_col="sample",
    spatial_key="spatial",
    region_col="lymph_region_type",
    n_cols=3,
    figsize_per_panel=5.0,
    point_size=10.0,            # 固定点大小
    adaptive_point_size=False,  # 关闭自适应
    alpha=0.95,
    suptitle="Spatial region map (fixed size = 10)",
    savepath=None,
    dpi=300,
    invert_y=False              # 若需要“影像方向”，可改 True
)
plt.show()


In [ ]:
# =========================
# Pretty region plotting: save ONE FIG per sample
# =========================
import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

OUTPUT_DIR = "/data/beifen/zhongmin/slide-tag/图/根据肿瘤细胞分区域/单独打印每个样本"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def _safe_name(s: str) -> str:
    return re.sub(r'[^-\w\._\u4e00-\u9fff]+', '_', str(s))

def _get_coords_key(adata, basis: str):
    """
    兼容 adata.obsm['spatial'] / adata.obsm['X_spatial'] 这种两种写法
    """
    if basis in adata.obsm:
        return basis
    xk = f"X_{basis}"
    if xk in adata.obsm:
        return xk
    raise KeyError(f"adata.obsm 里找不到 '{basis}' 或 '{xk}'")

def visualize_regions_save_each(
    adata,
    *,
    sample_col="sample",
    region_col="lymph_region_type",
    basis="spatial",
    palette=None,
    draw_order=("normal_distant", "normal_adjacent", "tumor_margin", "tumor_core"),
    unknown_label="Unknown",
    unknown_color="#D0D0D0",
    # style
    point_size=10,
    figsize=(8, 6),
    dpi=300,
    alpha=1.0,
    rasterized=True,
    invert_y=False,
    title=True,
    title_fontsize=12,
    # legend
    legend=True,
    legend_marker_size=6,
    legend_fontsize=8,
    # saving
    save_formats=("pdf", "png"),
    bbox_inches="tight",
    pad_inches=0.02,
):
    """
    每个 sample 保存一张：空间四区（lymph_region_type）
    """
    if sample_col not in adata.obs.columns:
        raise KeyError(f"adata.obs 缺少列：{sample_col}")
    if region_col not in adata.obs.columns:
        raise KeyError(f"adata.obs 缺少列：{region_col}")

    coords_key = _get_coords_key(adata, basis)

    if palette is None:
        palette = {
            "tumor_core":      "#E95C59",
            "tumor_margin":    "#E59CC4",
            "normal_adjacent": "#F1BB72",
            "normal_distant":  "#53A85F",
        }

    # 样本列表（稳定排序）
    samples = sorted(pd.Index(adata.obs[sample_col].astype(str)).unique())

    for s in samples:
        m = adata.obs[sample_col].astype(str).eq(s).to_numpy()
        if not m.any():
            continue

        ad = adata[m].copy()
        coords = np.asarray(ad.obsm[coords_key])[:, :2]

        # 规范化 region 标签
        reg = ad.obs[region_col].astype("string").str.strip()
        reg = reg.mask(reg.isna() | reg.eq("") | reg.str.lower().isin({"none", "nan", "<na>"}), unknown_label)
        labels = reg.astype(str).to_numpy()

        # 当前样本里出现了哪些类别（按 draw_order 排序）
        present = list(pd.unique(labels))
        ordered = [k for k in draw_order if k in present]
        if unknown_label in present and unknown_label not in ordered:
            ordered.append(unknown_label)
        # 可能出现其它奇怪标签，也兜底放最后
        leftovers = [k for k in present if k not in ordered]
        ordered = ordered + leftovers

        # 每类颜色
        pal = {k: palette.get(k, unknown_color) for k in ordered}
        pal[unknown_label] = pal.get(unknown_label, unknown_color)

        # 绘制顺序：背景先画，肿瘤后画（保证覆盖）
        # 这里直接按 ordered 循环画每一类
        fig, ax = plt.subplots(figsize=figsize)

        for k in ordered:
            mk = (labels == k)
            if not mk.any():
                continue
            ax.scatter(
                coords[mk, 0], coords[mk, 1],
                s=point_size,
                c=pal[k],
                linewidths=0,
                edgecolors="none",
                alpha=alpha,
                rasterized=rasterized
            )

        # 美化：无坐标轴、无边框
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_xlabel(""); ax.set_ylabel("")
        ax.set_aspect("equal", adjustable="box")
        for sp in ax.spines.values():
            sp.set_visible(False)
        if invert_y:
            ax.invert_yaxis()

        # 标题：只放 sample 和各类数量（可选）
        if title:
            vc = pd.Series(labels).value_counts()
            parts = [f"{k}:{int(vc.get(k, 0))}" for k in draw_order if k in vc.index]
            if unknown_label in vc.index:
                parts.append(f"{unknown_label}:{int(vc[unknown_label])}")
            ax.set_title(f"{s} | n={ad.n_obs} | " + "  ".join(parts), fontsize=title_fontsize)

        # 图例（右侧、无黑边）
        if legend:
            handles = [
                Line2D(
                    [0], [0],
                    marker="o",
                    linestyle="None",
                    color="none",
                    markerfacecolor=pal[k],
                    markeredgecolor="none",
                    markeredgewidth=0,
                    markersize=legend_marker_size,
                    label=str(k),
                )
                for k in ordered
            ]
            ax.legend(
                handles=handles,
                title="Region",
                loc="center left",
                bbox_to_anchor=(1.02, 0.5),
                frameon=False,
                fontsize=legend_fontsize,
                title_fontsize=legend_fontsize,
                handletextpad=0.3,
                labelspacing=0.4,
            )

        # 保存
        base = os.path.join(OUTPUT_DIR, f"{_safe_name(s)}_tumor_zones_{region_col}")
        for ext in save_formats:
            out = f"{base}.{ext}"
            fig.savefig(out, dpi=dpi, bbox_inches=bbox_inches, pad_inches=pad_inches)
            print(f"Saved: {out}")

        plt.close(fig)


# =========================
# Call
# =========================
visualize_regions_save_each(
    adata,
    sample_col="sample",
    region_col="lymph_region_type",
    basis="spatial",  # 若你的坐标在 X_spatial，可保持 basis="spatial"，函数会自动识别 X_spatial
    palette={
        "tumor_core":      "#E95C59",
        "tumor_margin":    "#E59CC4",
        "normal_adjacent": "#F1BB72",
        "normal_distant":  "#53A85F",
    },
    draw_order=("normal_distant", "normal_adjacent", "tumor_margin", "tumor_core"),
    point_size=10,
    figsize=(8, 6),
    dpi=300,
    legend=True,
    invert_y=False,          # 如需“影像方向”，改 True
    save_formats=("pdf", "png"),
)


5个区域的TLS数量图

In [ ]:
# ============================================
# Publication-style: TLS counts by region (5 groups)
# - unique TLS counts (NOT stacked by TLS type)
# - override region = "Granulomatosis" for samples with obs['subtype']=="Granulomatosis"
# - PURE WHITE background (NO grid lines)
# Save to: /data/beifen/zhongmin/slide-tag/图/根据肿瘤细胞分区域
# ============================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# 0) Config
# -----------------------------
OUTDIR = "/data/beifen/zhongmin/slide-tag/图/根据肿瘤细胞分区域"
os.makedirs(OUTDIR, exist_ok=True)

# cell-level columns in adata.obs
TLS_REGION_COL = "tls_region_tlsmode"         # TLS region assigned to each cell
TLS_ID_COL     = "tls_degrow_id_sample"       # unique TLS id (recommended)
SUBTYPE_COL    = "subtype"                    # sample subtype (Granulomatosis)

# 5-group region order (Granulomatosis + 4 tumor zoning regions)
REGION_ORDER = ["Granulomatosis", "tumor_core", "tumor_margin", "normal_adjacent", "normal_distant"]

# palette in the SAME ORDER as REGION_ORDER
REGION_COLORS_LIST = ['#E5D2DD', '#53A85F', '#F1BB72', '#F3B1A0', '#D6E7A3']
REGION_COLORS = {k: REGION_COLORS_LIST[i] for i, k in enumerate(REGION_ORDER)}

# output filenames
OUT_PDF = os.path.join(OUTDIR, "TLS_counts_by_region_5groups.pdf")
OUT_PNG = os.path.join(OUTDIR, "TLS_counts_by_region_5groups.png")
OUT_CSV = os.path.join(OUTDIR, "TLS_counts_by_region_5groups.csv")
OUT_TLS_LEVEL = os.path.join(OUTDIR, "TLS_level_region_5groups.csv")

# -----------------------------
# 1) Helpers
# -----------------------------
def _clean_str_series(s: pd.Series) -> pd.Series:
    s = s.astype("string").str.strip()
    s = s.mask(s.isna() | s.eq("") | s.str.lower().isin({"none", "nan", "<na>", "na"}), pd.NA)
    return s

def _mode_or_na(series: pd.Series):
    s = _clean_str_series(series).dropna()
    if s.empty:
        return pd.NA
    m = s.mode()
    return m.iloc[0] if not m.empty else pd.NA

def tls_level_table_region_only(
    adata,
    *,
    tls_id_col=TLS_ID_COL,
    tls_region_col=TLS_REGION_COL,
    subtype_col=SUBTYPE_COL,
):
    """
    Aggregate cell-level obs to TLS-level (one row per TLS).
    region/subtype take MODE after cleaning.
    Then apply: if subtype == "Granulomatosis" -> region_final="Granulomatosis"
    """
    for c in [tls_id_col, tls_region_col, subtype_col]:
        if c not in adata.obs.columns:
            raise KeyError(f"adata.obs 缺少列：{c}")

    df = adata.obs[[tls_id_col, tls_region_col, subtype_col]].copy()

    # keep valid TLS cells only
    tls_id = _clean_str_series(df[tls_id_col])
    keep = tls_id.notna()
    df = df.loc[keep].copy()
    df[tls_id_col] = tls_id.loc[keep]

    # stable mode
    df[tls_region_col] = df[tls_region_col].astype("string")
    df[subtype_col] = df[subtype_col].astype("string")

    g = df.groupby(df[tls_id_col], observed=True, sort=False)
    out = g.agg({
        tls_region_col: _mode_or_na,
        subtype_col: _mode_or_na,
    }).reset_index()

    out = out.rename(columns={
        tls_id_col: "tls_id",
        tls_region_col: "region_raw",
        subtype_col: "subtype",
    })

    subtype_clean = _clean_str_series(out["subtype"])
    region_clean  = _clean_str_series(out["region_raw"])

    is_gran = subtype_clean.fillna("").str.lower().eq("granulomatosis")
    region_final = region_clean.astype("string").copy()
    region_final = region_final.mask(is_gran, "Granulomatosis")

    out["region_final"] = region_final

    # drop TLS with no final region
    out = out.loc[_clean_str_series(out["region_final"]).notna()].copy()
    return out

def _apply_pub_style_no_grid(ax):
    """Pure white background, no gridlines, remove top/right spines."""
    ax.set_facecolor("white")
    ax.grid(False)  # <<< 关键：关闭网格线
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    # 可选：左/下边框也更细更干净
    ax.spines["left"].set_linewidth(1.0)
    ax.spines["bottom"].set_linewidth(1.0)
    ax.set_axisbelow(True)

def plot_tls_counts_by_region(counts: pd.Series, *, out_pdf, out_png, title=None):
    regions = counts.index.tolist()
    vals = counts.values.astype(int)

    fig, ax = plt.subplots(figsize=(7.8, 4.6))
    fig.patch.set_facecolor("white")  # <<< 画布背景纯白

    bar_colors = [REGION_COLORS.get(r, "#B0B0B0") for r in regions]
    x = np.arange(len(regions))

    ax.bar(x, vals, color=bar_colors, edgecolor="white", linewidth=1.2)

    # annotate counts
    ymax = max(vals) if len(vals) else 0
    pad = max(1, int(ymax * 0.02))
    for i, v in enumerate(vals):
        ax.text(i, v + pad, f"{v}", ha="center", va="bottom",
                fontsize=11, fontweight="bold", color="#222222")

    ax.set_xticks(x)
    ax.set_xticklabels(regions, rotation=20, ha="right")
    ax.set_ylabel("Number of TLS (unique TLS)")
    ax.set_xlabel("TLS region")
    if title is None:
        title = "TLS counts by region (unique TLS, 5 groups)"
    ax.set_title(title, fontsize=13, fontweight="bold")

    _apply_pub_style_no_grid(ax)

    plt.tight_layout()
    fig.savefig(out_pdf, bbox_inches="tight")      # PDF = 矢量
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)

# -----------------------------
# 2) Run
# -----------------------------
tls_lv = tls_level_table_region_only(
    adata,
    tls_id_col=TLS_ID_COL,
    tls_region_col=TLS_REGION_COL,
    subtype_col=SUBTYPE_COL,
)

# counts (unique TLS) by 5-group region
counts = tls_lv["region_final"].astype("string").value_counts()
counts = counts.reindex(REGION_ORDER, fill_value=0)

# save tables
counts.rename("tls_count").to_frame().to_csv(OUT_CSV)
tls_lv.to_csv(OUT_TLS_LEVEL, index=False)

# plot
plot_tls_counts_by_region(
    counts,
    out_pdf=OUT_PDF,
    out_png=OUT_PNG,
    title="TLS counts by region (unique TLS; Granulomatosis overridden)"
)

print("[Done] Saved:")
print(" ", OUT_PDF)
print(" ", OUT_PNG)
print(" ", OUT_CSV)
print(" ", OUT_TLS_LEVEL)
print("\nCounts:")
print(counts.to_string())


5个区域的不同TLS的占比

In [ ]:
# ============================================
# Publication-style TLS composition by region (5 groups, with Granulomatosis override)
# - 100% stacked proportion plot (stacked by TLS type colors)
# - stacked absolute count plot (stacked by TLS type colors)
# - Add "Granulomatosis" group: if TLS subtype == "Granulomatosis" -> region_final="Granulomatosis"
# Save to: /data/beifen/zhongmin/slide-tag/图/根据肿瘤细胞分区域
# ============================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# -----------------------------
# 0) Config
# -----------------------------
OUTDIR = "/data/beifen/zhongmin/slide-tag/图/根据肿瘤细胞分区域"
os.makedirs(OUTDIR, exist_ok=True)

REGION_COL  = "tls_region_tlsmode"
FENLEI_COL  = "tls_degrow_id_fenlei"
SUBTYPE_COL = "subtype"  # <<< 用于炎症/Granulomatosis 覆盖

# 优先用 sample-unique 的 TLS id
TLS_ID_COL = "tls_degrow_id_sample"  # fallback in code if not exist

# 5-group region order (Granulomatosis + 4 tumor zoning regions)
REGION_ORDER = ["Granulomatosis", "tumor_core", "tumor_margin", "normal_adjacent", "normal_distant"]

TLS_TYPES = ["Mature", "Conforming", "Deviating"]  # 只画这三类

# 颜色：只按 TLS type，不按 region
TYPE_COLORS = {
    "Mature": "#59A14F",
    "Conforming": "#4E79A7",
    "Deviating": "#E15759",
}

# -----------------------------
# 1) Helpers
# -----------------------------
def _clean_str_series(s: pd.Series) -> pd.Series:
    s = s.astype("string").str.strip()
    s = s.mask(s.isna() | s.eq("") | s.str.lower().isin({"none", "na", "nan", "<na>"}), pd.NA)
    return s

def _mode_or_nan(series: pd.Series):
    """众数（清理空/none/nan/na 后），为空则返回 np.nan。"""
    s = _clean_str_series(series).dropna()
    if s.empty:
        return np.nan
    m = s.mode()
    return m.iloc[0] if not m.empty else np.nan

def tls_level_table_unique_with_inflam_override(
    adata,
    *,
    tls_id_col: str,
    region_col: str,
    fenlei_col: str,
    subtype_col: str,
    infl_label: str = "Granulomatosis",
):
    """
    cell-level -> TLS-level（每个 TLS 一行）
    region/fenlei/subtype 取众数，降低单细胞噪声影响
    然后：若 subtype == infl_label，则 region_final 强制 = infl_label
    """
    need = [tls_id_col, region_col, fenlei_col, subtype_col]
    for c in need:
        if c not in adata.obs.columns:
            raise KeyError(f"adata.obs 缺少列：{c}")

    df = adata.obs[[tls_id_col, region_col, fenlei_col, subtype_col]].copy()

    # 清洗 TLS id
    tls_id = _clean_str_series(df[tls_id_col])
    keep = tls_id.notna()
    df = df.loc[keep].copy()
    df[tls_id_col] = tls_id.loc[keep]

    # 强制 string，避免 category 写回/聚合时问题
    df[region_col]  = df[region_col].astype("string")
    df[fenlei_col]  = df[fenlei_col].astype("string")
    df[subtype_col] = df[subtype_col].astype("string")

    g = df.groupby(df[tls_id_col], observed=True, sort=False)
    out = g.agg({
        region_col:  _mode_or_nan,
        fenlei_col:  _mode_or_nan,
        subtype_col: _mode_or_nan,
    }).reset_index()

    out = out.rename(columns={
        tls_id_col: "tls_id",
        region_col: "region_raw",
        fenlei_col: "fenlei",
        subtype_col: "subtype",
    })

    # 清洗
    region_raw = _clean_str_series(out["region_raw"]).astype("string")
    subtype    = _clean_str_series(out["subtype"]).astype("string")

    # 覆盖炎症/Granulomatosis
    is_inflam = subtype.fillna("").str.lower().eq(str(infl_label).lower())
    region_final = region_raw.copy()
    region_final = region_final.mask(is_inflam, infl_label)

    out["region_final"] = region_final

    # 丢掉 region_final 缺失的 TLS
    out = out.loc[_clean_str_series(out["region_final"]).notna()].copy()

    return out

def build_region_x_type_counts(tls_level_df, region_order, tls_types, region_col="region_final"):
    """
    返回 counts: DataFrame(index=region_order, columns=tls_types) 计数（TLS级）
    """
    df = tls_level_df.copy()
    df[region_col] = df[region_col].astype("string").str.strip()
    df["fenlei"]   = df["fenlei"].astype("string").str.strip()

    # 只保留三类 TLS
    df = df.loc[df["fenlei"].isin(tls_types)].copy()

    counts = pd.crosstab(df[region_col], df["fenlei"])

    # 补齐列/行并按指定顺序排序
    counts = counts.reindex(index=region_order, fill_value=0)
    counts = counts.reindex(columns=tls_types, fill_value=0)

    return counts

def _apply_pub_style(ax):
    """统一文章风格：去掉上右边框 + 轻网格。"""
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", linestyle="-", linewidth=0.6, alpha=0.25)
    ax.set_axisbelow(True)

def plot_stacked_percent(counts, outpath_pdf, outpath_png, colors, title):
    denom = counts.sum(axis=1).replace(0, np.nan)
    frac = counts.div(denom, axis=0).fillna(0.0)

    fig, ax = plt.subplots(figsize=(8.6, 4.8))
    x = np.arange(frac.shape[0])
    bottom = np.zeros(frac.shape[0])

    for t in frac.columns:
        vals = frac[t].to_numpy()
        ax.bar(
            x, vals, bottom=bottom,
            color=colors.get(t, "#999999"),
            edgecolor="white", linewidth=1.0,
            label=t
        )
        # 标注百分比（>=8% 才标）
        for i, (b, v) in enumerate(zip(bottom, vals)):
            if v >= 0.08:
                ax.text(i, b + v/2, f"{v*100:.0f}%",
                        ha="center", va="center",
                        fontsize=10, color="white", fontweight="bold")
        bottom += vals

    ax.set_xticks(x)
    ax.set_xticklabels(frac.index.tolist(), rotation=20, ha="right")
    ax.set_ylim(0, 1)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y*100:.0f}%"))
    ax.set_ylabel("Proportion of TLS (100% stacked)")
    ax.set_xlabel("TLS region")
    ax.set_title(title, fontsize=13, fontweight="bold")

    _apply_pub_style(ax)
    ax.legend(title="TLS type", frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")

    plt.tight_layout()
    fig.savefig(outpath_pdf, dpi=300, bbox_inches="tight")
    fig.savefig(outpath_png, dpi=300, bbox_inches="tight")
    plt.close(fig)

def plot_stacked_counts(counts, outpath_pdf, outpath_png, colors, title):
    fig, ax = plt.subplots(figsize=(8.6, 4.8))
    x = np.arange(counts.shape[0])
    bottom = np.zeros(counts.shape[0])

    for t in counts.columns:
        vals = counts[t].to_numpy()
        ax.bar(
            x, vals, bottom=bottom,
            color=colors.get(t, "#999999"),
            edgecolor="white", linewidth=1.0,
            label=t
        )
        bottom += vals

    # 顶部标注总数
    totals = counts.sum(axis=1).to_numpy()
    ymax = float(np.max(totals)) if len(totals) else 0.0
    for i, tot in enumerate(totals):
        if tot > 0:
            ax.text(i, tot + max(0.5, ymax*0.01), f"{int(tot)}",
                    ha="center", va="bottom",
                    fontsize=10, color="#222222", fontweight="bold")

    ax.set_xticks(x)
    ax.set_xticklabels(counts.index.tolist(), rotation=20, ha="right")
    ax.set_ylabel("Number of TLS (unique TLS)")
    ax.set_xlabel("TLS region")
    ax.set_title(title, fontsize=13, fontweight="bold")

    _apply_pub_style(ax)
    ax.legend(title="TLS type", frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")

    plt.tight_layout()
    fig.savefig(outpath_pdf, dpi=300, bbox_inches="tight")
    fig.savefig(outpath_png, dpi=300, bbox_inches="tight")
    plt.close(fig)

# -----------------------------
# 2) Main
# -----------------------------
def run_tls_region_figures_5groups(adata):
    # TLS id 列自动选择
    tls_id_col = TLS_ID_COL if TLS_ID_COL in adata.obs.columns else "tls_degrow_id"
    if tls_id_col not in adata.obs.columns:
        raise KeyError("adata.obs 中既没有 'tls_degrow_id_sample' 也没有 'tls_degrow_id'，无法做 TLS 级统计。")

    for c in [REGION_COL, FENLEI_COL, SUBTYPE_COL]:
        if c not in adata.obs.columns:
            raise KeyError(f"adata.obs 缺少列：{c}")

    # TLS-level 聚合 + Granulomatosis 覆盖
    tls_lv = tls_level_table_unique_with_inflam_override(
        adata,
        tls_id_col=tls_id_col,
        region_col=REGION_COL,
        fenlei_col=FENLEI_COL,
        subtype_col=SUBTYPE_COL,
        infl_label="Granulomatosis",
    )

    # region_final x type 计数（TLS级）
    counts = build_region_x_type_counts(
        tls_lv,
        region_order=REGION_ORDER,
        tls_types=TLS_TYPES,
        region_col="region_final"
    )

    # 输出路径
    out_percent_pdf = os.path.join(OUTDIR, "TLS_type_by_region_5groups_100pct_stacked.pdf")
    out_percent_png = os.path.join(OUTDIR, "TLS_type_by_region_5groups_100pct_stacked.png")
    out_count_pdf   = os.path.join(OUTDIR, "TLS_type_by_region_5groups_counts_stacked.pdf")
    out_count_png   = os.path.join(OUTDIR, "TLS_type_by_region_5groups_counts_stacked.png")

    # 画图（颜色仍只按 TLS type）
    plot_stacked_percent(
        counts,
        outpath_pdf=out_percent_pdf,
        outpath_png=out_percent_png,
        colors=TYPE_COLORS,
        title="TLS composition by region"
    )

    plot_stacked_counts(
        counts,
        outpath_pdf=out_count_pdf,
        outpath_png=out_count_png,
        colors=TYPE_COLORS,
        title="TLS composition by region"
    )

    print("[Done] Saved figures to:")
    print(" ", out_percent_pdf)
    print(" ", out_percent_png)
    print(" ", out_count_pdf)
    print(" ", out_count_png)

    return counts, tls_lv

# ---- run ----
counts, tls_level_df = run_tls_region_figures_5groups(adata)


不同肿瘤分期的TLS类型占比

In [ ]:
# ============================================
# Publication-style TLS composition by tumor stage (fixed order)
# - Stage from obs.subtype
# - TLS-level (unique TLS) aggregation: stage/fenlei take mode across cells
# - 100% stacked proportion plot + stacked counts plot
# - Force x-axis order: Granulomatosis, AIS, MIA, IA
# Save to: /data/beifen/zhongmin/slide-tag/图/根据肿瘤细胞分区域
# ============================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# -----------------------------
# 0) Config
# -----------------------------
OUTDIR = "/data/beifen/zhongmin/slide-tag/图/根据肿瘤细胞分区域"
os.makedirs(OUTDIR, exist_ok=True)

STAGE_COL  = "subtype"                 # 肿瘤分期/亚型（你说在 obs.subtype）
FENLEI_COL = "tls_degrow_id_fenlei"    # Mature/Conforming/Deviating
TLS_ID_COL = "tls_degrow_id_sample"    # fallback in code if not exist

TLS_TYPES = ["Mature", "Conforming", "Deviating"]  # 只画这三类

TYPE_COLORS = {
    "Mature": "#59A14F",
    "Conforming": "#4E79A7",
    "Deviating": "#E15759",
}

# >>> 关键：强制列顺序（你指定的顺序） <<<
STAGE_ORDER_FIXED = ["Granulomatosis", "AIS", "MIA", "IA"]

# -----------------------------
# 1) Helpers
# -----------------------------
def _clean_str_series(s: pd.Series) -> pd.Series:
    s = s.astype("string").str.strip()
    s = s.mask(s.isna() | s.eq("") | s.str.lower().isin({"none", "na", "nan", "<na>"}), pd.NA)
    return s

def _mode_or_nan(series: pd.Series):
    s = _clean_str_series(series).dropna()
    if s.empty:
        return np.nan
    m = s.mode()
    return m.iloc[0] if not m.empty else np.nan

def tls_level_table_unique_by_stage(
    adata,
    *,
    tls_id_col: str,
    stage_col: str,
    fenlei_col: str,
):
    """
    cell-level -> TLS-level（每个 TLS 一行）
    stage/fenlei 取众数，降低单细胞噪声影响
    """
    need = [tls_id_col, stage_col, fenlei_col]
    for c in need:
        if c not in adata.obs.columns:
            raise KeyError(f"adata.obs 缺少列：{c}")

    df = adata.obs[[tls_id_col, stage_col, fenlei_col]].copy()

    # 清洗 TLS id
    tls_id = _clean_str_series(df[tls_id_col])
    keep = tls_id.notna()
    df = df.loc[keep].copy()
    df[tls_id_col] = tls_id.loc[keep]

    df[stage_col]  = df[stage_col].astype("string")
    df[fenlei_col] = df[fenlei_col].astype("string")

    g = df.groupby(df[tls_id_col], observed=True, sort=False)
    out = g.agg({
        stage_col:  _mode_or_nan,
        fenlei_col: _mode_or_nan,
    }).reset_index()

    out = out.rename(columns={
        tls_id_col: "tls_id",
        stage_col: "stage_final",
        fenlei_col: "fenlei",
    })

    # 丢掉 stage_final 缺失的 TLS
    out = out.loc[_clean_str_series(out["stage_final"]).notna()].copy()

    return out

def build_group_x_type_counts(tls_level_df, group_order, tls_types, group_col="stage_final"):
    df = tls_level_df.copy()
    df[group_col] = df[group_col].astype("string").str.strip()
    df["fenlei"]  = df["fenlei"].astype("string").str.strip()

    # 只保留三类 TLS
    df = df.loc[df["fenlei"].isin(tls_types)].copy()

    counts = pd.crosstab(df[group_col], df["fenlei"])

    # 补齐列/行并按指定顺序排序
    counts = counts.reindex(index=group_order, fill_value=0)
    counts = counts.reindex(columns=tls_types, fill_value=0)

    return counts

def _apply_pub_style(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", linestyle="-", linewidth=0.6, alpha=0.25)
    ax.set_axisbelow(True)

def plot_stacked_percent(counts, outpath_pdf, outpath_png, colors, title, xlabel):
    denom = counts.sum(axis=1).replace(0, np.nan)
    frac = counts.div(denom, axis=0).fillna(0.0)

    fig, ax = plt.subplots(figsize=(8.6, 4.8))
    x = np.arange(frac.shape[0])
    bottom = np.zeros(frac.shape[0])

    for t in frac.columns:
        vals = frac[t].to_numpy()
        ax.bar(
            x, vals, bottom=bottom,
            color=colors.get(t, "#999999"),
            edgecolor="white", linewidth=1.0,
            label=t
        )
        # 标注百分比（>=8% 才标）
        for i, (b, v) in enumerate(zip(bottom, vals)):
            if v >= 0.08:
                ax.text(i, b + v/2, f"{v*100:.0f}%",
                        ha="center", va="center",
                        fontsize=10, color="white", fontweight="bold")
        bottom += vals

    ax.set_xticks(x)
    ax.set_xticklabels(frac.index.tolist(), rotation=20, ha="right")
    ax.set_ylim(0, 1)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y*100:.0f}%"))
    ax.set_ylabel("Proportion of TLS (100% stacked)")
    ax.set_xlabel(xlabel)
    ax.set_title(title, fontsize=13, fontweight="bold")

    _apply_pub_style(ax)
    ax.legend(title="TLS type", frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")

    plt.tight_layout()
    fig.savefig(outpath_pdf, dpi=300, bbox_inches="tight")
    fig.savefig(outpath_png, dpi=300, bbox_inches="tight")
    plt.close(fig)

def plot_stacked_counts(counts, outpath_pdf, outpath_png, colors, title, xlabel):
    fig, ax = plt.subplots(figsize=(8.6, 4.8))
    x = np.arange(counts.shape[0])
    bottom = np.zeros(counts.shape[0])

    for t in counts.columns:
        vals = counts[t].to_numpy()
        ax.bar(
            x, vals, bottom=bottom,
            color=colors.get(t, "#999999"),
            edgecolor="white", linewidth=1.0,
            label=t
        )
        bottom += vals

    # 顶部标注总数
    totals = counts.sum(axis=1).to_numpy()
    ymax = float(np.max(totals)) if len(totals) else 0.0
    for i, tot in enumerate(totals):
        if tot > 0:
            ax.text(i, tot + max(0.5, ymax*0.01), f"{int(tot)}",
                    ha="center", va="bottom",
                    fontsize=10, color="#222222", fontweight="bold")

    ax.set_xticks(x)
    ax.set_xticklabels(counts.index.tolist(), rotation=20, ha="right")
    ax.set_ylabel("Number of TLS (unique TLS)")
    ax.set_xlabel(xlabel)
    ax.set_title(title, fontsize=13, fontweight="bold")

    _apply_pub_style(ax)
    ax.legend(title="TLS type", frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")

    plt.tight_layout()
    fig.savefig(outpath_pdf, dpi=300, bbox_inches="tight")
    fig.savefig(outpath_png, dpi=300, bbox_inches="tight")
    plt.close(fig)

# -----------------------------
# 2) Main
# -----------------------------
def run_tls_stage_figures_fixed_order(adata):
    tls_id_col = TLS_ID_COL if TLS_ID_COL in adata.obs.columns else "tls_degrow_id"
    if tls_id_col not in adata.obs.columns:
        raise KeyError("adata.obs 中既没有 'tls_degrow_id_sample' 也没有 'tls_degrow_id'，无法做 TLS 级统计。")

    for c in [STAGE_COL, FENLEI_COL]:
        if c not in adata.obs.columns:
            raise KeyError(f"adata.obs 缺少列：{c}")

    # TLS-level 聚合
    tls_lv = tls_level_table_unique_by_stage(
        adata,
        tls_id_col=tls_id_col,
        stage_col=STAGE_COL,
        fenlei_col=FENLEI_COL,
    )

    # >>> 强制 stage 顺序（你指定的） <<<
    stage_order = STAGE_ORDER_FIXED

    # stage_final x type 计数
    counts = build_group_x_type_counts(
        tls_lv,
        group_order=stage_order,
        tls_types=TLS_TYPES,
        group_col="stage_final"
    )

    # 输出路径
    out_percent_pdf = os.path.join(OUTDIR, "TLS_type_by_stage_100pct_stacked.pdf")
    out_percent_png = os.path.join(OUTDIR, "TLS_type_by_stage_100pct_stacked.png")
    out_count_pdf   = os.path.join(OUTDIR, "TLS_type_by_stage_counts_stacked.pdf")
    out_count_png   = os.path.join(OUTDIR, "TLS_type_by_stage_counts_stacked.png")

    # 画图
    plot_stacked_percent(
        counts,
        outpath_pdf=out_percent_pdf,
        outpath_png=out_percent_png,
        colors=TYPE_COLORS,
        title="TLS composition by tumor stage",
        xlabel="Tumor stage (obs.subtype)"
    )

    plot_stacked_counts(
        counts,
        outpath_pdf=out_count_pdf,
        outpath_png=out_count_png,
        colors=TYPE_COLORS,
        title="TLS composition by tumor stage",
        xlabel="Tumor stage (obs.subtype)"
    )

    print("[Done] Saved figures to:")
    print(" ", out_percent_pdf)
    print(" ", out_percent_png)
    print(" ", out_count_pdf)
    print(" ", out_count_png)

    return counts, tls_lv, stage_order

# ---- run ----
counts, tls_level_df, stage_order = run_tls_stage_figures_fixed_order(adata)
print("[Stage order] ", stage_order)
print(counts)


5个区域的细胞类型占比

In [ ]:
# ============================================================
# TLS celltype composition across 5 region groups (stacked bar)
# - Groups: Granulomatosis + tumor_core + tumor_margin + normal_adjacent + normal_distant
# - TLS membership: cells with valid TLS_ID_COL
# - Cell types: adata.obs['celltype_4_ZZM']
# - Output: proportion stacked bar (publication-style) + CSV
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# 0) Config
# -----------------------------
OUTDIR = "/data/beifen/zhongmin/slide-tag/图/根据肿瘤细胞分区域"
os.makedirs(OUTDIR, exist_ok=True)

TLS_ID_COL     = "tls_degrow_id_sample"
TLS_REGION_COL = "tls_region_tlsmode"
SUBTYPE_COL    = "subtype"
CELLTYPE_COL   = "celltype_4_ZZM"  # <<< 确认你的真实列名

REGION_ORDER = ["Granulomatosis", "tumor_core", "tumor_margin", "normal_adjacent", "normal_distant"]

CELLTYPE_PALETTE = [
    '#E5D2DD', '#53A85F', '#F1BB72', '#F3B1A0', '#D6E7A3', '#57C3F3', '#476D87',
    '#E95C59', '#E59CC4', '#AB3282', '#23452F', '#BD956A', '#8C549C', '#585658',
    '#9FA3A8', '#E0D4CA', '#5F3D69', '#C5DEBA', '#58A4C3', '#E4C755', '#F7F398',
    '#AA9A59', '#E63863', '#E39A35', '#C1E6F3', '#6778AE', '#91D0BE', '#B53E2B',
    '#712820', '#DCC1DD', '#CCE0F5', '#CCC9E6', '#625D9E', '#68A180', '#3A6963',
    '#968175'
]

OUT_PDF = os.path.join(OUTDIR, "TLS_celltype_composition_by_region5_stacked_proportion.pdf")
OUT_PNG = os.path.join(OUTDIR, "TLS_celltype_composition_by_region5_stacked_proportion.png")
OUT_CSV = os.path.join(OUTDIR, "TLS_celltype_composition_by_region5_proportions.csv")
OUT_COUNTS_CSV = os.path.join(OUTDIR, "TLS_celltype_composition_by_region5_counts.csv")

TOP_N_CELLTYPES = None  # e.g. 20; set None to keep all

# >>> 关键：在这里调图的宽高（单位：英寸） <<<
FIGSIZE = (8, 5)   # 例如：更高一些；你也可以试 (12, 8) / (10, 7) 等

# -----------------------------
# 1) Helpers
# -----------------------------
def _clean_str_series(s: pd.Series) -> pd.Series:
    s = s.astype("string").str.strip()
    s = s.mask(s.isna() | s.eq("") | s.str.lower().isin({"none", "nan", "<na>", "na"}), pd.NA)
    return s

def _apply_pub_style(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", linestyle="-", linewidth=0.6, alpha=0.25)
    ax.set_axisbelow(True)

def build_tls_region5_labels(adata, *, tls_region_col, subtype_col):
    reg = _clean_str_series(adata.obs[tls_region_col]).fillna(pd.NA)
    sub = _clean_str_series(adata.obs[subtype_col]).fillna(pd.NA)
    is_gran = sub.fillna("").str.lower().eq("granulomatosis")
    reg5 = reg.astype("string").copy()
    reg5 = reg5.mask(is_gran, "Granulomatosis")
    return reg5

def stacked_bar_proportion(df_prop, color_map, out_pdf, out_png, title, *, figsize=FIGSIZE):
    """
    df_prop: index=region5, columns=celltypes, values=fractions (0-1)
    """
    fig, ax = plt.subplots(figsize=figsize)  # <<< 关键：用传入的 figsize

    x = np.arange(df_prop.shape[0])
    bottom = np.zeros(df_prop.shape[0])

    for ct in df_prop.columns:
        vals = df_prop[ct].to_numpy()
        ax.bar(
            x, vals, bottom=bottom,
            color=color_map.get(ct, "#B0B0B0"),
            edgecolor="white", linewidth=0.8,
            label=ct
        )
        bottom += vals

    ax.set_xticks(x)
    ax.set_xticklabels(df_prop.index.tolist(), rotation=20, ha="right")
    ax.set_ylim(0, 1)
    ax.set_ylabel("Cell-type proportion within TLS cells")
    ax.set_xlabel("TLS region group (5 groups)")
    ax.set_title(title, fontsize=13, fontweight="bold")

    _apply_pub_style(ax)

    ax.legend(
        title=CELLTYPE_COL,
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=False,
        fontsize=8,
        title_fontsize=9,
        ncol=1
    )

    plt.tight_layout()
    fig.savefig(out_pdf, bbox_inches="tight")     # PDF = 矢量
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)

# -----------------------------
# 2) Main
# -----------------------------
def run_tls_celltype_composition_by_region5(adata):
    for c in [TLS_ID_COL, TLS_REGION_COL, SUBTYPE_COL, CELLTYPE_COL]:
        if c not in adata.obs.columns:
            raise KeyError(f"adata.obs 缺少列：{c}")

    tls_id = _clean_str_series(adata.obs[TLS_ID_COL])
    m_tls = tls_id.notna()
    if not np.any(m_tls):
        raise ValueError("没有检测到任何 TLS 细胞（tls_degrow_id_sample 全空）。")

    ad_tls = adata[m_tls].copy()

    reg5 = build_tls_region5_labels(ad_tls, tls_region_col=TLS_REGION_COL, subtype_col=SUBTYPE_COL)
    reg5 = _clean_str_series(reg5).fillna(pd.NA)

    ct = _clean_str_series(ad_tls.obs[CELLTYPE_COL]).fillna("Unknown").astype(str)
    reg5_str = reg5.astype("string")
    keep = reg5_str.isin(REGION_ORDER).to_numpy()
    if not keep.any():
        raise ValueError("TLS 细胞中没有任何 region 落在 5 组（REGION_ORDER）里。请检查 tls_region_tlsmode/subtype。")

    reg5_str = reg5_str[keep].astype(str)
    ct = ct[keep]

    counts = pd.crosstab(reg5_str, ct).reindex(index=REGION_ORDER, fill_value=0)

    if TOP_N_CELLTYPES is not None and counts.shape[1] > TOP_N_CELLTYPES:
        totals = counts.sum(axis=0).sort_values(ascending=False)
        keep_ct = totals.index[:TOP_N_CELLTYPES]
        other_ct = [c for c in counts.columns if c not in keep_ct]
        counts2 = counts[keep_ct].copy()
        counts2["Other"] = counts[other_ct].sum(axis=1)
        counts = counts2

    denom = counts.sum(axis=1).replace(0, np.nan)
    prop = counts.div(denom, axis=0).fillna(0.0)

    celltypes_sorted = sorted(prop.columns.tolist())
    if len(CELLTYPE_PALETTE) < len(celltypes_sorted):
        raise ValueError(
            f"配色数量不足：palette={len(CELLTYPE_PALETTE)} < celltypes={len(celltypes_sorted)}。"
            " 请增加配色或设置 TOP_N_CELLTYPES。"
        )
    color_map = {ct_name: CELLTYPE_PALETTE[i] for i, ct_name in enumerate(celltypes_sorted)}
    if "Other" in color_map:
        color_map["Other"] = "#D0D0D0"

    counts.to_csv(OUT_COUNTS_CSV)
    prop.to_csv(OUT_CSV)

    stacked_bar_proportion(
        prop[celltypes_sorted],
        color_map=color_map,
        out_pdf=OUT_PDF,
        out_png=OUT_PNG,
        title="TLS cell-type composition across 5 region groups (proportion; TLS cells only)",
        figsize=FIGSIZE,  # <<< 关键：这里也可以传你想要的大小
    )

    print("[Done] Saved:")
    print(" ", OUT_PDF)
    print(" ", OUT_PNG)
    print(" ", OUT_COUNTS_CSV)
    print(" ", OUT_CSV)
    return counts, prop

# ---- run ----
counts_df, prop_df = run_tls_celltype_composition_by_region5(adata)


不同TLS类型的细胞类型占比

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUTDIR = "/data/beifen/zhongmin/slide-tag/图/根据肿瘤细胞分区域"
os.makedirs(OUTDIR, exist_ok=True)

TLS_ID_COL = "tls_degrow_id_sample"
TLS_CLASS_COL = "tls_degrow_id_fenlei"
SUBTYPE_COL = "subtype"
CELLTYPE_COL = "celltype_4_ZZM"

TLS_GROUP_ORDER = ["Deviating_Tumor", "Deviating_Inflammation", "Conforming", "Mature"]

INFLAM_SUBTYPES = {
    "Granulomatosis",
    "Inflammation",
    "Inflammatory",
}

CELLTYPE_COLOR_MAP = {
    "Malignant cells": "#968175",
    "Fibroblasts": "#968175",
    "AT2": "#968175",
    "Endothelial cells": "#476D87",
    "AT1": "#968175",
    "B cells": "#AB3282",
    "CD4 T cells": "#712820",
    "CD8 T cells": "#57C3F3",
    "Plasma cells": "#CCC9E6",
    "Multiciliated": "#91D0BE",
    "NK cells": "#AA9A59",
    "Mast cells": "#BD956A",
    "SMC": "#CCE0F5",
    "DC2": "#DCC1DD",
    "DC1": "#C1E6F3",
    "Macrophages": "#E59CC4",
    "Treg": "#E4C755",
    "FDC": "#23452F",
    "Tfh": "#000000",
    "LAMP3_DC": "#585658",
    "Pericyte": "#F3B1A0",
    "CD4 Trm": "#53A85F",
    "CD8 Trm": "#E63863",
}

DEFAULT_COLOR = "#B0B0B0"
OTHER_COLOR = "#D0D0D0"

OUT_PDF = os.path.join(OUTDIR, "TLS_celltype_composition_DeviatingSplit_stacked_proportion.pdf")
OUT_PNG = os.path.join(OUTDIR, "TLS_celltype_composition_DeviatingSplit_stacked_proportion.png")
OUT_CSV = os.path.join(OUTDIR, "TLS_celltype_composition_DeviatingSplit_proportions.csv")
OUT_COUNTS_CSV = os.path.join(OUTDIR, "TLS_celltype_composition_DeviatingSplit_counts.csv")

TOP_N_CELLTYPES = None
FIGSIZE = (8.6, 4.2)

# 图例参数（你后续只要调这里）
LEGEND_FONTSIZE = 10
LEGEND_BBOX_TO_ANCHOR = (1.01, 1.0)
LEGEND_LABELSPACING = 0.4
LEGEND_HANDLELENGTH = 1.2
LEGEND_HANDLETEXTPAD = 0.5

# 给右侧图例预留空间（0.78~0.86 之间调，越小留白越大）
TIGHT_LAYOUT_RECT = [0, 0, 0.82, 1]

def _clean_str_series(s: pd.Series) -> pd.Series:
    s = s.astype("string").str.strip()
    s = s.mask(s.isna() | s.eq("") | s.str.lower().isin({"none", "nan", "<na>", "na"}), pd.NA)
    return s

def _apply_pub_style(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", linestyle="-", linewidth=0.6, alpha=0.25)
    ax.set_axisbelow(True)

def build_tls_group_with_deviating_split(tls_class: pd.Series, subtype: pd.Series) -> pd.Series:
    cls = _clean_str_series(tls_class)
    sub = _clean_str_series(subtype)

    inflam_set = {str(x).strip().lower() for x in INFLAM_SUBTYPES}
    is_inflam = sub.fillna("").str.lower().isin(inflam_set)

    out = cls.astype("string").copy()
    is_dev = out.fillna("").str.lower().eq("deviating")

    out = out.mask(is_dev & (~is_inflam), "Deviating_Tumor")
    out = out.mask(is_dev & (is_inflam), "Deviating_Inflammation")

    out = out.astype("string")
    out = out.mask(out.fillna("").str.lower().eq("conforming"), "Conforming")
    out = out.mask(out.fillna("").str.lower().eq("mature"), "Mature")
    return out

def _order_celltypes_for_plot(celltypes_present):
    preferred = [ct for ct in CELLTYPE_COLOR_MAP.keys() if ct in celltypes_present]
    remaining = sorted([ct for ct in celltypes_present if ct not in set(preferred)])
    ordered = preferred + remaining
    if "Other" in ordered:
        ordered = [ct for ct in ordered if ct != "Other"] + ["Other"]
    return ordered

def stacked_bar_proportion(df_prop, color_map, out_pdf, out_png, title, *, figsize=FIGSIZE):
    fig, ax = plt.subplots(figsize=figsize)

    x = np.arange(df_prop.shape[0])
    bottom = np.zeros(df_prop.shape[0])

    for ct in df_prop.columns:
        vals = df_prop[ct].to_numpy()
        ax.bar(
            x, vals, bottom=bottom,
            color=color_map.get(ct, DEFAULT_COLOR),
            edgecolor="white", linewidth=0.8,
            label=ct
        )
        bottom += vals

    ax.set_xticks(x)
    ax.set_xticklabels(df_prop.index.tolist(), rotation=20, ha="right")
    ax.set_ylim(0, 1)
    ax.set_ylabel("Cell-type proportion within TLS cells")
    ax.set_xlabel("TLS group (Deviating split by subtype)")
    ax.set_title(title, fontsize=13, fontweight="bold")

    _apply_pub_style(ax)

    ax.legend(
        title=None,
        bbox_to_anchor=LEGEND_BBOX_TO_ANCHOR,
        loc="upper left",
        frameon=False,
        fontsize=LEGEND_FONTSIZE,
        ncol=1,
        borderaxespad=0.0,
        labelspacing=LEGEND_LABELSPACING,
        handlelength=LEGEND_HANDLELENGTH,
        handletextpad=LEGEND_HANDLETEXTPAD
    )

    # ✅ 给右侧图例预留空间，避免超出画布
    plt.tight_layout(rect=TIGHT_LAYOUT_RECT)

    fig.savefig(out_pdf, bbox_inches="tight")
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)

def run_tls_celltype_composition_deviating_split(adata):
    for c in [TLS_ID_COL, TLS_CLASS_COL, SUBTYPE_COL, CELLTYPE_COL]:
        if c not in adata.obs.columns:
            raise KeyError(f"adata.obs 缺少列：{c}")

    tls_id = _clean_str_series(adata.obs[TLS_ID_COL])
    m_tls = tls_id.notna()
    if not np.any(m_tls):
        raise ValueError("没有检测到任何 TLS 细胞（tls_degrow_id_sample 全空）。")

    ad_tls = adata[m_tls].copy()

    tls_group = build_tls_group_with_deviating_split(
        ad_tls.obs[TLS_CLASS_COL],
        ad_tls.obs[SUBTYPE_COL],
    )
    tls_group = _clean_str_series(tls_group)

    celltype = _clean_str_series(ad_tls.obs[CELLTYPE_COL]).fillna("Unknown").astype(str)

    keep = tls_group.astype("string").isin(TLS_GROUP_ORDER).to_numpy()
    if not keep.any():
        raise ValueError(
            f"TLS 细胞中没有任何分组落在 TLS_GROUP_ORDER={TLS_GROUP_ORDER} 里。\n"
            f"请检查 '{TLS_CLASS_COL}' 及 '{SUBTYPE_COL}'。"
        )

    tls_group = tls_group[keep].astype(str)
    celltype = celltype[keep]

    counts = pd.crosstab(tls_group, celltype)
    counts = counts.reindex(index=TLS_GROUP_ORDER, fill_value=0)

    if TOP_N_CELLTYPES is not None and counts.shape[1] > TOP_N_CELLTYPES:
        totals = counts.sum(axis=0).sort_values(ascending=False)
        keep_ct = totals.index[:TOP_N_CELLTYPES]
        other_ct = [c for c in counts.columns if c not in keep_ct]
        counts2 = counts[keep_ct].copy()
        counts2["Other"] = counts[other_ct].sum(axis=1)
        counts = counts2

    denom = counts.sum(axis=1).replace(0, np.nan)
    prop = counts.div(denom, axis=0).fillna(0.0)

    counts.to_csv(OUT_COUNTS_CSV)
    prop.to_csv(OUT_CSV)

    ordered_celltypes = _order_celltypes_for_plot(prop.columns.tolist())

    color_map = dict(CELLTYPE_COLOR_MAP)
    color_map["Other"] = OTHER_COLOR

    stacked_bar_proportion(
        prop[ordered_celltypes],
        color_map=color_map,
        out_pdf=OUT_PDF,
        out_png=OUT_PNG,
        title="TLS cell-type composition",
        figsize=FIGSIZE
    )

    print("[Done] Saved:")
    print(" ", OUT_PDF)
    print(" ", OUT_PNG)
    print(" ", OUT_COUNTS_CSV)
    print(" ", OUT_CSV)

    print("\n[Group sizes (TLS cells)]:")
    print(counts.sum(axis=1).to_string())

    print("\n[Subtypes present (top 30)]:")
    print(adata.obs[SUBTYPE_COL].value_counts(dropna=False).head(30).to_string())

    unknown_cts = [ct for ct in ordered_celltypes if ct not in CELLTYPE_COLOR_MAP and ct != "Other"]
    if len(unknown_cts) > 0:
        print("\n[Warning] 以下 celltype 没在 CELLTYPE_COLOR_MAP 中，将使用默认灰色：")
        print(" ", ", ".join(unknown_cts))

    return counts, prop

counts_df, prop_df = run_tls_celltype_composition_deviating_split(adata)
